# SnowIn core workflow

This self-contained notebook introduces the main xarray-native SnowIn APIs without requiring a downloaded NISAR product. It demonstrates phase-to-dSWE conversion, reference phase, corrections, support layers, temporal accumulation, metrics, and basic plotting.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
import xarray as xr

import snowin
from snowin import (
    accumulate_dswe,
    build_support_dataset,
    compose_support_mask,
    compute_dswe,
    compute_metrics,
    reference_phase,
    summarize_support,
)
from snowin.corrections import PhaseCorrectionConfig, apply_phase_corrections
from snowin.reference import apply_reference_phase

print(f"SnowIn {snowin.__version__}")

## 1. Create a normalized phase pair

SnowIn's canonical phase is `secondary_minus_reference`, and angles are stored in radians.

In [ ]:
rng = np.random.default_rng(7)
ny, nx = 60, 80
y = np.arange(ny, dtype=float)
x = np.arange(nx, dtype=float)
coords = {"y": y, "x": x}

incidence_values = np.broadcast_to(
    np.deg2rad(32.0 + 5.0 * np.sin(x[None, :] / 18.0)), (ny, nx)
)
incidence = xr.DataArray(
    incidence_values,
    dims=("y", "x"),
    coords=coords,
    name="incidence_angle",
    attrs={"units": "rad", "incidence_angle_reference": "local"},
)
phase_values = 0.8 * np.sin(x[None, :] / 14.0) + 0.3 * np.cos(y[:, None] / 11.0)
phase_values += 0.04 * rng.normal(size=(ny, nx))
phase_values[:4, :6] = np.nan
phase = xr.DataArray(
    phase_values,
    dims=("y", "x"),
    coords=coords,
    name="phase",
    attrs={"units": "rad", "phase_difference_definition": "secondary_minus_reference"},
)
pair = xr.Dataset(
    {"phase": phase, "incidence_angle": incidence},
    attrs={
        "phase_difference_definition": "secondary_minus_reference",
        "reference_time": "2025-01-01T00:00:00Z",
        "secondary_time": "2025-01-13T00:00:00Z",
    },
)
pair

## 2. Compute pairwise dSWE

The canonical kernel returns dSWE in metres and preserves the grid and metadata.

In [ ]:
wavelength_m = 0.238403545
dswe = compute_dswe(pair["phase"], pair["incidence_angle"], wavelength_m=wavelength_m)
display(dswe)
assert dswe.attrs["units"] == "m"
assert dswe.dims == pair["phase"].dims

## 3. Apply explicit reference phase

Reference inputs are supplied by the caller. SnowIn does not choose stations or dates.

In [ ]:
station_ids = xr.DataArray(
    ["station_a", "station_b", "station_c", "station_d"], dims="station"
)
observed = xr.DataArray(
    [0.37, 0.41, np.nan, 0.36], dims="station", attrs={"units": "rad"}
)
expected = xr.DataArray(
    [0.10, 0.12, 0.20, 0.11], dims="station", attrs={"units": "rad"}
)
weights = xr.DataArray([0.8, 0.6, 0.4, 0.9], dims="station", attrs={"units": "1"})
exclusion_reason = xr.DataArray(
    ["eligible", "eligible", "missing_station_phase", "eligible"], dims="station"
)

referenced = reference_phase(
    pair,
    observed,
    expected,
    weights,
    contributor_id=station_ids,
    exclusion_reason=exclusion_reason,
)
print("reference offset [rad]:", referenced.attrs["phase_reference_offset_rad"])
print("eligible contributors:", referenced.attrs["phase_reference_eligible_count"])
referenced_dswe = compute_dswe(
    referenced["phase_referenced"],
    referenced["incidence_angle"],
    wavelength_m=wavelength_m,
)
assert referenced.attrs["phase_reference_status"] == "SUPPORTED"

The older NumPy compatibility helper is still available for simple diagnostics, but it is not an implicit default in the xarray workflow.

In [ ]:
legacy_reference = apply_reference_phase(pair["phase"].values, strategy="median_valid")
legacy_reference.diagnostics

## 4. Apply an explicitly signed phase correction

Corrections require a declared sign and remain separate from reference phase.

In [ ]:
correction_layer = np.full((ny, nx), 0.02)
correction = apply_phase_corrections(
    pair["phase"].values,
    wet_tropo=correction_layer,
    config=PhaseCorrectionConfig(apply_wet_tropo=True, wet_tropo_sign="subtract"),
)
print(correction.diagnostics)
assert np.allclose(
    correction.phase_corrected[10:, 10:],
    pair["phase"].values[10:, 10:] - 0.02,
    equal_nan=True,
)

## 5. Build and summarize named support layers

In [ ]:
product_valid = np.isfinite(pair["phase"]).rename("product_valid")
geometry_valid = (
    (pair["incidence_angle"] >= 0) & (pair["incidence_angle"] < math.pi / 2)
).rename("geometry_valid")
reference_supported = xr.ones_like(product_valid, dtype=bool).rename(
    "reference_supported"
)
pairwise_supported = np.isfinite(referenced_dswe).rename("pairwise_supported")
support = build_support_dataset(
    product_valid=product_valid,
    geometry_valid=geometry_valid,
    reference_supported=reference_supported,
    pairwise_supported=pairwise_supported,
)
support_mask = compose_support_mask(
    support,
    ["product_valid", "geometry_valid", "reference_supported", "pairwise_supported"],
)
summarize_support(support)

## 6. Accumulate a chronological path

In [ ]:
def temporal_edge(values, reference_time, secondary_time):
    values = values.rename("dswe").copy()
    values.attrs = {
        **values.attrs,
        "units": "m",
        "quantity": "pairwise_dSWE",
        "phase_difference_definition": "secondary_minus_reference",
    }
    return xr.Dataset(
        {"dswe": values},
        attrs={
            "temporal_edge": "reference_to_secondary",
            "reference_time": reference_time,
            "secondary_time": secondary_time,
        },
    )


edge_1 = temporal_edge(referenced_dswe, "2025-01-01T00:00:00Z", "2025-01-13T00:00:00Z")
edge_2 = temporal_edge(
    referenced_dswe * 0.5, "2025-01-13T00:00:00Z", "2025-01-25T00:00:00Z"
)
path = accumulate_dswe([edge_1, edge_2])
path

## 7. Compute metrics and make quick plots

In [ ]:
observed_dswe = referenced_dswe.rename("observed_dswe")
estimated_dswe = (referenced_dswe + 0.005).rename("estimated_dswe")
observed_dswe.attrs = {"units": "m"}
estimated_dswe.attrs = {"units": "m"}
metrics = compute_metrics(observed_dswe, estimated_dswe, support=support_mask)
metrics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
pair["phase"].plot(ax=axes[0], cmap="viridis")
axes[0].set_title("Canonical phase [rad]")
referenced_dswe.plot(ax=axes[1], cmap="RdBu_r")
axes[1].set_title("Referenced dSWE [m]")
support_mask.plot(ax=axes[2], cmap="Greens")
axes[2].set_title("Explicit support")
plt.show()

assert path.attrs["temporal_edge_count"] == 2
assert (
    metrics.attrs["support_policy"]
    == "finite_observed_and_estimated_and_explicit_support"
)
print("Core SnowIn workflow checks passed.")